### Aufgabe: Frequenzgang und Sweep-Messung

Wir betrachten ein System mit der folgenden Z-Transformationsfunktion:

$$
H(z) = 1 - z^{-m}
$$

**a)** Berechnen Sie den Frequenz-, Phasen- und Impulsverlauf des Systems.  
(Mit Stift und Papier)  
(Machen Sie es wirklich — eine ähnliche Aufgabe könnte auch in der mündlichen Prüfung vorkommen.)

**b)** Berechnen Sie den Amplituden- und Phasengang mit der Python-Funktion `freqz()` aus dem Modul `scipy.signal` und stellen Sie die Ergebnisse (in linearer Skalierung) für den Bereich  
$0 \leq \Omega < 2 \pi$ dar.  
Setzen Sie $m = 1$.  
Überprüfen Sie Ihre Berechnungen aus Teil a) dieser Aufgabe.

**c)** Berechnen Sie die Impulsantwort mit der Funktion `impz()` (bereitgestellt in `helper_functions.py`) und plotten Sie die Ergebnisse.  
Erstellen Sie ein Nullstellen-Pol-Diagramm mit der Funktion `zplane()` (ebenfalls in `helper_functions.py`).  
Erklären Sie, um welchen Filtertyp es sich handelt.

**d)** In der Praxis kann ein Frequenzgang häufig mithilfe sogenannter Sweep-Messungen bestimmt werden.  
Bei Sweep-Messungen wird ein System mit Schwingungen unterschiedlicher Frequenzen angeregt.  
Ein LTI-System reagiert auf eine Schwingung mit derselben Frequenz, jedoch mit unterschiedlicher Phase und Amplitude.  
Wenn die Frequenz über den interessierenden Teil des Spektrums ($0 \dots \pi$) variiert wird,  
können der Amplituden- und Phasengang bestimmt werden.

In dieser Aufgabe soll der Amplitudengang mit einem Anregungssignal der Form

$$
x(k) = \cos(\Omega_i k)
$$

gemessen werden.  
Vergleichen Sie die Ergebnisse mit denen aus Teil b).

**e)** Regen Sie das System mit einer Sprungfolge $\varepsilon(k)$ an und zeigen Sie, dass die erste Differenz der Sprungantwort der Impulsantwort entspricht.  
Hinweis: Verwenden Sie die Funktion `np.diff()`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from helper_functions import impz, zplane
# BEGIN SOLUTION
# Filter / parameters
H = np.array([1, -1], dtype=float)   # FIR: H(z) = 1 - z^{-1}
A = np.array([1.0], dtype=float)
N = 32

# #####  Part b: Amplitude and phase response  #####
w, Hf = signal.freqz(H, A, worN=256, whole=True)  # w in [0, 2π)

# Amplitude
plt.figure()
plt.plot(w/(2*np.pi), np.abs(Hf))
plt.title('Amplitude Response')
plt.xlabel(r'$\Omega/2\pi$')
plt.ylabel('Amplitude')

# Phase (note: -angle as in MATLAB definition)
plt.figure()
plt.plot(w/(2*np.pi), -np.angle(Hf)/np.pi*180)
plt.yticks([-90, -45, 0, 45, 90])
plt.title('Phase Response')
plt.xlabel(r'$\Omega/2\pi$')
plt.ylabel('Phase in Degrees')

# #####  Part c: Impulse response, pole-zero plot  #####
h = impz(H, A, 4)   
lh = len(h)

# Impulse response (stem)
plt.figure()
(markerline, stemlines, baseline) = plt.stem(np.arange(lh), h)
plt.axis([-0.1, 3.1, -1, 1])
plt.plot([-0.1, 3.1], [0, 0], 'k')
plt.title('Impulse Response')
plt.xlabel('k')
plt.ylabel('Amplitude')

# Pole-zero plot
zplane(H, A)  

# #####  Part d: Sweep measurement – cosine excitation #####
lf = len(H)
k_sweep = np.arange(0, N - 1 + 2*(lf - 1) + 1)  # include endpoint like MATLAB
n_sweep = np.arange(0, N//2)                    # 0 .. N/2-1
erg = []

for n in n_sweep:
    sweep = np.cos(2*np.pi*n/N * k_sweep)
    foo = np.convolve(H, sweep)
    # remove transients at both ends (lf-1 samples)
    core = foo[(lf-1):len(foo)-(lf-1)]
    erg.append(np.max(np.abs(core)))
erg = np.array(erg)

# Comparison: theoretical amplitude response (standard freqz: w in [0, π])
w_th, H_th = signal.freqz(H, A, worN=N, whole=False)

plt.figure()
plt.plot(2*n_sweep/N, erg, 'r+')
plt.plot(w_th/np.pi, np.abs(H_th), 'b-')
plt.title('+ Sweep Measurement, - Theoretical Amplitude Response')
plt.xlabel(r'$\Omega/\pi$')
plt.ylabel('Amplitude')

# #####  Part e: Sweep measurement – exponential (complex) excitation #####
erg = []
for n in n_sweep:
    sweep = np.exp(1j*2*np.pi*n/N * k_sweep)
    foo = np.convolve(H, sweep)
    core = foo[(lf-1):len(foo)-(lf-1)]
    erg.append(np.max(np.abs(core)))
erg = np.array(erg)

plt.figure()
plt.plot(2*n_sweep/N, erg, 'r+')
plt.plot(w_th/np.pi, np.abs(H_th), 'b-')
plt.title('+ Sweep Measurement, - Theoretical Amplitude Response')
plt.xlabel(r'$\Omega/\pi$')
plt.ylabel('Amplitude')

# #####  Part f: Step response  #####
epsilon = np.r_[np.zeros(10), np.ones(10)]
step_response = np.convolve(h, epsilon)
hdiff = np.diff(step_response)

# Step sequence
plt.figure()
plt.stem(np.arange(len(epsilon)) - 10, epsilon)
plt.title('Step Sequence')
plt.xlabel('k')
plt.ylabel(r'$\epsilon(k)$')

# Step response
plt.figure()
plt.stem(np.arange(len(step_response)) - 10, step_response)
plt.plot([-10, 15], [0, 0], 'k')
plt.title('Step Response')
plt.xlabel('k')
plt.ylabel('Amplitude')

# Impulse response (from diff of step response)
plt.figure()
plt.stem(np.arange(len(hdiff)) - 10, hdiff)
plt.plot([-10, 15], [0, 0], 'k')
plt.title('Impulse Response')
plt.xlabel('k')
plt.ylabel('Amplitude')

plt.show()
# END SOLUTION

### Lösung: Frequenzgang und Sweep-Messung

**a)**

$$
H(z) = 1 - z^{-m}
$$

Die **Amplitudenantwort** $\left|H(e^{j \Omega})\right|$ des Systems ergibt sich zu:

$$
H(e^{j \Omega}) = 1 - e^{-j \Omega m}
$$

$$
\begin{aligned}
\left|H(e^{j \Omega})\right|
  &= \left| 1 - e^{-j \Omega m} \right| \\
  &= \left|  e^{-j \frac{\Omega}{2} m} \left( e^{j \frac{\Omega}{2} m} - e^{-j  \frac{\Omega}{2} m} \right) \right| \\
  &= \underbrace{\left|  e^{-j \frac{\Omega}{2} m} \right|}_{1} \left|\left( e^{j \frac{\Omega}{2} m} - e^{-j \frac{\Omega}{2} m} \right) \right| \\
  &= \left| \left( e^{j \frac{\Omega}{2} m} - e^{-j \frac{\Omega}{2} m} \right) \right| \\
  &= \left| 2 j \sin \left( \frac{\Omega}{2} m\right) \right| \\
\left|H(e^{j \Omega})\right|  &= 2 \left| \sin \left( \frac{\Omega}{2} m\right) \right|
\end{aligned}
$$

Die **Phasenantwort** $b(\Omega)$ ergibt sich gemäß der Definition:  
(Achtung: Die Phasenantwort ist hier mit einem negativen Vorzeichen definiert, sodass sich die Gruppenlaufzeit durch die positive Ableitung der Phasenantwort ergibt  
$\tau_g(\Omega) = \frac{\mathrm{d}\, b(\Omega)}{\mathrm{d}\, \Omega}$.  
Normalerweise wird die Phasenantwort positiv definiert, sodass zur Bestimmung der Gruppenlaufzeit ein negatives Vorzeichen erforderlich ist.)

$$
H(e^{j \Omega}) = \left| H(e^{j \Omega}) \right| e^{- j b(\Omega) }
$$

und wird wie folgt berechnet:

$$
\begin{aligned}
b(\Omega) &= - \arg\!\left(H(e^{j\Omega})\right) \\
          &= - \arctan\!\left(\frac{\Im\{H(e^{j\Omega})\}}{\Re\{H(e^{j\Omega})\}}\right)
\end{aligned}
$$

Zu diesem Zweck ist es zweckmäßig, die Übertragungsfunktion analog zu der obigen Rechnung umzuformen:

$$
\begin{aligned}
H(e^{j \Omega}) &= 1 - e^{-j \Omega m} \\
                &= e^{-j \frac{\Omega}{2} m}  \left[ e^{j \frac{\Omega}{2} m} - e^{-j \frac{\Omega}{2} m}\right] \\
        &= \left[ \cos \left( \frac{\Omega}{2} m \right) - j \sin \left( \frac{\Omega}{2} m \right) \right]
        \left[ 2 j \sin \left( \frac{\Omega}{2} m \right) \right] \\
        &= 2 \sin^2 \left( \frac{\Omega}{2} m \right) + j\, 2 \sin \left( \frac{\Omega}{2} m \right)
        \cos \left( \frac{\Omega}{2} m \right)
\end{aligned}
$$

$$
\begin{aligned}
b(\Omega) &= - \arctan \frac{\Im\{H(e^{j\Omega})\}}{\Re\{H(e^{j\Omega})\}} \\
          &= - \arctan \frac{ 2 \sin \left( \tfrac{\Omega}{2} m \right)
        \cos \left( \tfrac{\Omega}{2} m \right) }{ 2 \sin^2 \left( \tfrac{\Omega}{2} m \right)  } \\
          &= - \arctan \frac{ \cos \left( \tfrac{\Omega}{2} m \right) }{ \sin \left( \tfrac{\Omega}{2} m \right)  } \\
          &= - \arctan \cot \left( \tfrac{\Omega}{2} m \right)  \\
          & \downarrow \quad \text{mit } \arctan x = \tfrac{\pi}{2} - \operatorname{arccot} x \\
      &= - \tfrac{\pi}{2} + \operatorname{arccot}\, \cot \left( \tfrac{\Omega}{2} m \right)  \\
b(\Omega) &= - \tfrac{\pi}{2} + \tfrac{\Omega}{2} m
\end{aligned}
$$


**b)**

Wenn die Python-Funktion `scipy.signal.freqz()` verwendet wird, werden der Amplituden- und Phasengang gemäß den folgenden Abbildungen dargestellt:

<img src="../pngs/ldsv2a18_b1.png" width="250">  
<img src="../pngs/ldsv2a18_b2.png" width="250">  

Es zeigt sich, dass der Kurvenverlauf mit der analytischen Betrachtung aus Teil a) übereinstimmt.

**c)**

Die Impulsantwort, die mit der Hilfsfunktion `impz()` (bereitgestellt in `helper_functions.py`) berechnet wird, ist in den folgenden Abbildungen dargestellt.  
Ihr Verlauf ergibt sich direkt aus der Z-Transformationsfunktion:

$$
h(k) = 1 \delta_0(k) - 1 \delta_0(k-m)
$$

<img src="../pngs/ldsv2a18_c1.png" width="250">  
<img src="../pngs/ldsv2a18_c2.png" width="250">  

Der Pol im Ursprung des Pol-Nullstellen-Diagramms ergibt sich aus:

$$
H(z) = 1 - z^{-m} = \frac{z^m - 1}{z^m}
$$

Es handelt sich um einen **Hochpassfilter**, da die tiefen Frequenzen durch die Nullstellen an der Position $z_0 = 1$ unterdrückt werden.

**d)**

Der Amplitudengang, der mithilfe einer Sweep-Messung bestimmt wurde, stimmt mit der theoretischen Berechnung überein.

<img src="../pngs/ldsv2a18_d.png" width="250">  

**e)**

Die folgende Abbildung zeigt die Sprungantwort des Systems.  
Neben der eigentlichen Antwort ist auch die Transientenantwort zu erkennen, die durch die endliche Länge der Sprungfolge $\epsilon(k)$ entsteht.

<img src="../pngs/ldsv2a18_e1.png" width="250">  
<img src="../pngs/ldsv2a18_e2.png" width="250">  

Durch die Bildung der ersten Differenz (mithilfe von `np.diff()`) kann die Impulsantwort des Systems erneut dargestellt werden.

<img src="../pngs/ldsv2a18_e3.png" width="250">  